In [ ]:
from astropy.io import fits 
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
from typing import Literal

In [ ]:
with fits.open('/home/bekah/m3-pipeline-dev/data/moon_dss_ghost_corr.fits') as hdul:
    data = hdul[0].data

band = data[3, :, :] 

# moon row = 
row = band[3402, :]

In [ ]:
# KERNEL 

def make_kernel(radius=60, core_sigma=5.0, tail_sigma=10.0, tail_weight=0.18):
    '''Symmetric 1D scatter kernel: narrow core + broad tail, normalized to sum to 1
    (i.e. energy-conserving: all the light that leaves a source pixel lands somewhere in-frame).
    '''
    x = np.arange(-radius, radius + 1)
    core = np.exp(-0.5 * (x / core_sigma) ** 2)
    tail = np.exp(-0.5 * (x / tail_sigma) ** 2)
    k = (1 - tail_weight) * core + tail_weight * tail
    k /= k.sum()
    return k

kernel = make_kernel(radius=90)


In [ ]:
# Wiener deconvoltuion 

def wiener_deconvolve(Y, kernel, K):
    '''Per-row Wiener deconvolution via FFT. Rows are edge-padded by the kernel radius
    before transforming (and cropped back after) to reduce wraparound artifacts, since our
    forward model uses 'nearest' boundary handling rather than true circular convolution.'''
    radius = (len(kernel) - 1) // 2
    pad = radius
    Npad = Y.shape[1] + 2 * pad
    # Embed the kernel into an Npad-length array with its peak at index 0 (required for
    # a correctly-phased circular/FFT-based convolution/deconvolution).
    k_full = np.zeros(Npad)
    k_full[:radius + 1] = kernel[radius:]
    k_full[-radius:] = kernel[:radius]
    Hf = np.fft.rfft(k_full)

    out = np.zeros_like(Y)
    for i, row in enumerate(Y):
        row_pad = np.pad(row, pad, mode="edge")
        Yf = np.fft.rfft(row_pad)
        Xf = np.conj(Hf) / (np.abs(Hf) ** 2 + K) * Yf
        x = np.fft.irfft(Xf, n=Npad)
        out[i] = x[pad:pad + Y.shape[1]]
    return out

K = 0.01

wiener = wiener_deconvolve(band, kernel, K)

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d

# img: 2D image, shape (rows, columns)
img = img.astype(np.float32)

# Estimate the smooth horizontal illumination profile for each row
sigma = 50  # adjust to the spatial scale of the scattered light
illumination = gaussian_filter1d(img, sigma=sigma, axis=1)

# Avoid division by very small values
eps = 1e-6
corrected = img / np.maximum(illumination, eps)

# If you want to preserve the overall intensity scale:
corrected *= np.mean(illumination)

# Alternatively, "subtract" the estimated light and add it back
# with a uniform row intensity:
# row_mean = np.mean(illumination, axis=1, keepdims=True)
# corrected = img / np.maximum(illumination, eps) * row_mean


In [ ]:
# Richard Lucy 

def build_S(kernel, N, calib_noise_std=0.0, seed=None):
    '''Build a banded matrix from a (possibly noisy/measured) 1D kernel.
    Column j = kernel centered at row j, truncated at the image edges (so edge columns
    naturally have slightly less than unit sum -- this mimics a real truncated PSF).'''
    rng = np.random.default_rng(seed)
    k_meas = kernel + rng.normal(0, calib_noise_std, kernel.shape)
    k_meas = np.clip(k_meas, 0, None)
    k_meas /= k_meas.sum()
    radius = (len(k_meas) - 1) // 2
    S = np.zeros((N, N))
    for j in range(N):
        lo, hi = max(0, j - radius), min(N, j + radius + 1)
        k_lo, k_hi = radius - (j - lo), radius + (hi - j)
        S[lo:hi, j] = k_meas[k_lo:k_hi]
    return S

S = build_S(kernel, 320, calib_noise_std=0.01, seed=3)

def rl_deconvolve(Y, S, n_iter=60, eps=1e-8):
    '''Richardson-Lucy-style multiplicative update for observed = S @ true (per row).
    Non-negative by construction, and flux-conserving when S's columns sum to ~1.'''
    col_sums = S.sum(axis=0)
    X = np.clip(Y.copy(), eps, None)
    for _ in range(n_iter):
        pred = np.clip((S @ X.T).T, eps, None)
        ratio = Y / pred
        correction = (S.T @ ratio.T).T / col_sums[None, :]
        X = X * correction
    return X

rl = rl_deconvolve(band, S, n_iter=20)



In [ ]:
from scipy.signal import convolve2d as conv2
from skimage import color, data, restoration


psf = np.ones((50, 50)) / 25

deconvolved_RL = restoration.richardson_lucy(band, psf, num_iter=30)


In [ ]:
# rolling ball 

ball = restoration.rolling_ball(band, radius=80)

In [ ]:
plt.figure(figsize=(9, 3.5))

b = 3402
plt.plot(rl[b, :], label="RL", lw=1.5)
plt.plot(wiener[b, :], label="Wiener", alpha=0.6)
plt.plot(band[b, :], label="OG", lw=1.2)
plt.plot(deconvolved_RL[b, :], label="scipy rl", lw=1.2)
plt.plot(ball[b, :], label="ball ", lw=1.2)

plt.ylim(-100, 3000)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fits.writeto('rl.fits', rl, overwrite=True)
fits.writeto('wiener.fits', wiener, overwrite=True)